### Supplement 1: Basic chat completion (`1-basic.py`)

Send two messages to the model and print the text it writes back.

**The key point:** `client.chat.completions.create(...)` sends your `messages` to OpenAI as JSON. The SDK turns the JSON reply into a `ChatCompletion` object, and the text you want is at `completion.choices[0].message.content`.

```
messages            a list of dicts, written by you
   │  client.chat.completions.create(model=..., messages=messages)
   ▼
OpenAI's server     runs the model and replies with JSON text
   │  the SDK turns that JSON into an object
   ▼
completion          a ChatCompletion object
   └─ .choices[0].message.content    a plain str: the limerick
```

Deep dive: [Exhaustive_1-basic.ipynb](Exhaustive_1-basic.ipynb).

#### 1. Imports and the API key
`load_dotenv()` looks for a `.env` file, starting in this folder and moving up, and loads its variables (including `OPENAI_API_KEY`) into the environment, where `os.getenv` can read them.

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

import json  # added for the prints below

load_dotenv()

print("OPENAI_API_KEY loaded:", os.getenv("OPENAI_API_KEY") is not None)

OPENAI_API_KEY loaded: True


#### 2. The client
`OpenAI` is a class, a blueprint. `client` is the object built from it: it holds your API key and the server address, and every request goes through it.

In [2]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("type(client):   ", type(client))
print("client.base_url:", client.base_url)

type(client):    <class 'openai.OpenAI'>
client.base_url: https://api.openai.com/v1/


#### 3. The messages
A plain Python list of dicts; no SDK class is involved. Each dict has a `role` and some `content`:
- `system`: instructions for how the model should behave.
- `user`: the request itself.

`1-basic.py` writes this list inside the `create()` call. Here it gets its own variable so it can be printed.

In [3]:
messages = [
    {"role": "system", "content": "You're a helpful assistant."},
    {
        "role": "user",
        "content": "Write a limerick about the Python programming language.",
    },
]

print("type(messages):   ", type(messages))
print("len(messages):    ", len(messages))
print("type(messages[0]):", type(messages[0]))
print()
print(json.dumps(messages, indent=2))  # the same list as JSON text, the form it travels in

type(messages):    <class 'list'>
len(messages):     2
type(messages[0]): <class 'dict'>

[
  {
    "role": "system",
    "content": "You're a helpful assistant."
  },
  {
    "role": "user",
    "content": "Write a limerick about the Python programming language."
  }
]


#### 4. The API call
`create()` sends `model` and `messages` to OpenAI and waits for the reply. It returns a `ChatCompletion` object, not a string. `gpt-5-nano` is a reasoning model: it thinks privately before it writes, so the call can take a while.

In [4]:
completion = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
)

In [5]:
print(type(completion))
# model_dump_json() is a Pydantic method: it writes the whole object out as JSON text.
print(completion.model_dump_json(indent=2))

<class 'openai.types.chat.chat_completion.ChatCompletion'>
{
  "id": "chatcmpl-EPu20m1eUyzdQScfOMylxCqEvKqkX",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "There once was a language called Python\nIts syntax was friendly, a trusty Python\nIndentation kept blocks neat\nBatteries included, can't be beat\nFor tasks small and grand, we love Python",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1789842292,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 3314,
    "prompt_tokens": 26,
    "total_tokens": 3340,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens":

#### 5. Getting the text out
The text is four levels down, and each level is a different SDK class until `.content`, which is a plain `str`.
- `choices` is a list because you can ask for several alternative replies (the `n` parameter). By default you get one, at index `0`.
- `finish_reason` is `'stop'` when the model ended the reply by itself.

In [6]:
# type(x).__name__ is the class name without its module path
print("completion.choices                    :", type(completion.choices).__name__, f"({len(completion.choices)} item)")
print("completion.choices[0]                 :", type(completion.choices[0]).__name__)
print("completion.choices[0].message         :", type(completion.choices[0].message).__name__)
print("completion.choices[0].message.content :", type(completion.choices[0].message.content).__name__)
print()
print("finish_reason:", completion.choices[0].finish_reason)
print("role:         ", completion.choices[0].message.role)

completion.choices                    : list (1 item)
completion.choices[0]                 : Choice
completion.choices[0].message         : ChatCompletionMessage
completion.choices[0].message.content : str

finish_reason: stop
role:          assistant


In [7]:
response = completion.choices[0].message.content
print(response)

There once was a language called Python
Its syntax was friendly, a trusty Python
Indentation kept blocks neat
Batteries included, can't be beat
For tasks small and grand, we love Python


#### 6. Token usage
You pay per token (a chunk of text, roughly ¾ of an English word) for what the model reads (`prompt_tokens`) and what it writes (`completion_tokens`). `completion_tokens` includes the hidden reasoning tokens, which is why it is far larger than a five-line limerick needs.

In [8]:
usage = completion.usage
reasoning = usage.completion_tokens_details.reasoning_tokens

print("prompt_tokens (read):       ", usage.prompt_tokens)
print("completion_tokens (written):", usage.completion_tokens)
print("   hidden reasoning:        ", reasoning)
print("   the visible limerick:    ", usage.completion_tokens - reasoning)
print("total_tokens:               ", usage.total_tokens)

prompt_tokens (read):        26
completion_tokens (written): 3314
   hidden reasoning:         3264
   the visible limerick:     50
total_tokens:                3340
